We test this on ETHUSDT, to ensure our feature engineering adaption, from the 28 features laid out in, [Joint-Embedding Predictive Learning of Latent Market States in U.S. Equities](https://openreview.net/pdf?id=BZfkxSasd3) (these are described in Table 7) are being properly adapted for the crypto market.

First, we set up imports and import the `ETHUSDT_hourly.csv` file as a dataframe.

In [1]:
import pandas as pd
import numpy as np

# prevent division by zero on forward-filled flat bars
EPSILON = 1e-8

# import the ETHUSDT data
df = pd.read_csv('hourly-gap-filling/ETHUSDT_hourly.csv')

First we cut out the bloat from the original dataset.

In [2]:
# Rebuild the datetime index
df.set_index(pd.to_datetime(df['open_time_ms'], unit='ms'), inplace=True)

# Define the bloat columns to purge
bloat_cols = [
    'open_time', 'open_time_ms', 
    'close_time', 'close_time_ms', 
    'interval',
    'quote_volume', 'trades', 
    'taker_buy_base_volume', 'taker_buy_quote_volume'
]

# Safely drop the bloat columns if they exist in the CSV
cols_to_drop = [col for col in bloat_cols if col in df.columns]
df_clean = df.drop(columns=cols_to_drop)

# Verify the result is exactly 5 columns (OHLCV) on a DatetimeIndex
print(df_clean.info())
print("\nFirst 3 rows:")
print(df_clean.head(3))

<class 'pandas.DataFrame'>
DatetimeIndex: 78093 entries, 2017-08-17 04:00:00 to 2026-07-15 00:00:00
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    78093 non-null  float64
 1   high    78093 non-null  float64
 2   low     78093 non-null  float64
 3   close   78093 non-null  float64
 4   volume  78093 non-null  float64
 5   symbol  78093 non-null  str    
dtypes: float64(5), str(1)
memory usage: 4.2 MB
None

First 3 rows:
                       open    high    low   close     volume   symbol
open_time_ms                                                          
2017-08-17 04:00:00  301.13  302.57  298.0  301.61  125.66877  ETHUSDT
2017-08-17 05:00:00  301.61  303.28  300.0  303.10  377.67246  ETHUSDT
2017-08-17 06:00:00  302.40  304.44  301.9  302.68  303.86672  ETHUSDT


Now we define the function to set up the returns features using the windows outlined in the paper, but adapted for the crypto market (which trades 24/7 unlike tradtional markets).

In [3]:
def construct_returns_features(df):
    """
    Constructs log returns and horizon structure features.
    """
    # 1d, 5d, 21d, 63d, 126d
    windows = {
        '1d': 24, 
        '5d': 120, 
        '21d': 504, 
        '63d': 1512, 
        '126d': 3024
    }
    
    # Trailing Log Returns
    for name, h in windows.items():
        df[f'log_ret_{name}'] = np.log(df['close'] / df['close'].shift(h))
        
    # Intraday (Intra-bar) and Overnight (Inter-bar)
    df['intraday_ret'] = np.log(df['close'] / (df['open'] + EPSILON))
    
    # we drop overnight gap here because it is not relevant for crypto, we have full 24 hour cycles for our coins
    
    # Momentum (126d return - 21d return)
    df['mom_6_1'] = df['log_ret_126d'] - df['log_ret_21d']
    
    return df

Next we construct the OHLC candle geometry features.

In [4]:
def construct_geometry_features(df):
    """
    Constructs price geometry and wick/shadow features.
    """
    # Log Range
    df['hl_log_range'] = np.log((df['high']) / (df['low']))
    
    # Normalized Body to Range
    body_abs = np.abs(np.log((df['close']) / (df['open'])))
    range_eps = np.log((df['high']) / (df['low']))

    # only add epsilon to this value as the U.S. equities paper describes to preven a possible division by zero error
    df['body_to_range'] = body_abs / (range_eps + EPSILON)
    
    # Upper and Lower Shadows (Wicks)
    max_oc = np.maximum(df['open'], df['close'])
    min_oc = np.minimum(df['open'], df['close'])
    
    df['upper_shadow'] = np.log((df['high']) / (max_oc))
    df['lower_shadow'] = np.log((min_oc) / (df['low']))
    
    return df

Next we calculate the volatility & Regime features.

In [ ]:
def construct_volatility_features(df):
    """
    Constructs rolling realized volatility and EWMA volatility.
    Fixes the overlapping autocorrelation bias by using discrete 24-hour jumps.
    """
    # 1. REALIZED VOLATILITY (Horizontal Non-Overlapping Std Dev)
    # Collect the last N non-overlapping 24-hour returns
    ret_10d_matrix = pd.concat([df['log_ret_1d'].shift(24 * i) for i in range(10)], axis=1)
    ret_21d_matrix = pd.concat([df['log_ret_1d'].shift(24 * i) for i in range(21)], axis=1)
    ret_63d_matrix = pd.concat([df['log_ret_1d'].shift(24 * i) for i in range(63)], axis=1)
    
    # Calculate Std Dev across the columns (axis=1). No scaling factor needed!
    df['rvol_10'] = ret_10d_matrix.std(axis=1, skipna=False)
    df['rvol_21'] = ret_21d_matrix.std(axis=1, skipna=False)
    df['rvol_63'] = ret_63d_matrix.std(axis=1, skipna=False)
    
    # Regime Ratio
    df['rvol_ratio'] = df['rvol_10'] / (df['rvol_63'] + EPSILON)
    
    # 2. EWMA VOLATILITY (Interleaved Daily Grouping)
    ret_squared = df['log_ret_1d'] ** 2
    
    # Group by the hour of the day to create 24 isolated daily tracks, 
    # apply the halflife, and drop the grouping index to merge it back
    # This effectively locks ewma volatility to timezones ie it is calculated over the same clock-face hour each day
    ewma_10 = ret_squared.groupby(df.index.hour).apply(
        lambda x: x.ewm(halflife=10, min_periods=10).mean()
    ).reset_index(level=0, drop=True)
    
    ewma_20 = ret_squared.groupby(df.index.hour).apply(
        lambda x: x.ewm(halflife=20, min_periods=20).mean()
    ).reset_index(level=0, drop=True)
    
    # Sort index just in case the groupby scrambled the chronological order
    df['ewma_vol_hl10'] = np.sqrt(ewma_10).sort_index()
    df['ewma_vol_hl20'] = np.sqrt(ewma_20).sort_index()
    
    return df

Next calculate the Volume & liquidity features.

In [6]:
def construct_liquidity_features(df):
    """
    Constructs liquidity and volume dynamics, including Amihud Illiquidity.
    """
    # Dollar Volume
    dollar_vol = df['volume'] * df['close']
    
    # Log Volume and Log Dollar Volume
    df['log_volume'] = np.log(df['volume'] + 1)
    df['log_dollar_volume'] = np.log(dollar_vol + 1)
    
    # Relative Dollar Volume (21d / 504h)
    mean_dvol_21 = dollar_vol.rolling(window=504).mean()
    df['rel_dvol_21'] = dollar_vol / (mean_dvol_21 + EPSILON)
    
    # Dollar Volume Z-Score (21d / 504h)
    std_dvol_21 = dollar_vol.rolling(window=504).std()
    df['dvol_z_21'] = (dollar_vol - mean_dvol_21) / (std_dvol_21 + EPSILON)
    
    # Amihud Illiquidity (Price impact per dollar traded)
    df['amihud_illiq_1'] = np.abs(df['log_ret_1d']) / (dollar_vol + EPSILON)

    # rolling 21d
    df['amihud_illiq_21'] = df['amihud_illiq_1'].rolling(window=504).mean()
    
    return df

Now we will test the symbol dependent features on just the ETH/USDT pair to ensure we are on the right track and our features are being calculated correctly.

In [7]:
df_features = df_clean.copy()

# Execute Pipeline
df_features = construct_returns_features(df_features)
df_features = construct_geometry_features(df_features)
df_features = construct_volatility_features(df_features)
df_features = construct_liquidity_features(df_features)

# Print a block of data that includes at least one synthetic gap (e.g., row 49091)
# to verify the geometric and liquidity features output 0 instead of NaN/Inf
print(df_features[['close', 'body_to_range', 'rvol_ratio', 'log_volume', 'amihud_illiq_1']].iloc[49088:49095])

# Print the overall NaN count to ensure NaNs are ONLY at the very beginning of the dataset 
# (due to the 3024-hour rolling windows)
print("\nTotal NaNs per column:")
print(df_features.isna().sum())

                       close  body_to_range  rvol_ratio  log_volume  \
open_time_ms                                                          
2023-03-24 12:00:00  1789.52       0.000000    0.960741    0.000000   
2023-03-24 13:00:00  1789.52       0.000000    0.993316    0.000000   
2023-03-24 14:00:00  1763.12       0.807086    0.964042   10.997770   
2023-03-24 15:00:00  1767.01       0.214628    1.044119   10.620448   
2023-03-24 16:00:00  1762.76       0.436730    1.188224    9.953379   
2023-03-24 17:00:00  1737.51       0.661456    1.153682   11.057683   
2023-03-24 18:00:00  1756.03       0.599437    0.927493   10.669167   

                     amihud_illiq_1  
open_time_ms                         
2023-03-24 12:00:00    2.163474e+06  
2023-03-24 13:00:00    2.370401e+06  
2023-03-24 14:00:00    2.151452e-10  
2023-03-24 15:00:00    5.365908e-10  
2023-03-24 16:00:00    9.886684e-10  
2023-03-24 17:00:00    4.527254e-10  
2023-03-24 18:00:00    3.420192e-10  

Total NaNs per co

Optionally, save to a CSV for further inspection.

In [8]:
df_features.to_csv('ETHUSDT_test_out.csv', index=True)